# 06 影视剧/小说素材抽取与打标

流程：把你自有授权的剧本/小说 txt 放入 `raw_txt/` → 抽取对话/分段 → 本地 4B（或服务器 32B）生成人格标签 → 合并进统一 JSONL。

**版权提醒**：仅使用你拥有授权或公有领域的素材；受版权保护的影视剧/小说不得入库。

In [1]:
# 1) 准备：raw_txt/ 下放 txt 文件（UTF-8），自动按「」「」或换行分段
import os, re, pathlib
RAW = os.path.expanduser("~/Desktop/Parallax项目/SML构建部署/语料数据集/raw_txt")
os.makedirs(RAW, exist_ok=True)
segs=[]
for p in pathlib.Path(RAW).glob("*.txt"):
    t=p.read_text(encoding="utf-8")
    t=re.sub(r'<.*?>','',t,flags=re.S)
    quotes=re.findall(r'[「“]([^」”]{15,300})[」”]',t) or re.findall(r'\n([^\n]{15,300})\n',t)
    for x in quotes:
        x=re.sub(r'\s+','',x).strip()
        if 15<=len(x)<=300: segs.append({"file":p.name,"text":x})
print("抽取片段:",len(segs),"（放入 raw_txt 后重跑本 cell）")

抽取片段: 0 （放入 raw_txt 后重跑本 cell）


In [2]:
# 2) 本地 4B 打标（人格特征提取；批量跑可用服务器 32B 替换模型路径）
import time, torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
P=os.path.expanduser("~/models/parallax/Qwen3-4B-Instruct-2507")
tok=AutoTokenizer.from_pretrained(P)
m=AutoModelForCausalLM.from_pretrained(P,dtype=torch.float16).to("mps").eval()
PROMPT="你是人格特征提取专家。基于文本推断说话者的人格特征，只输出 JSON："
'{"ocean":{"openness":0-5,"conscientiousness":0-5,"extraversion":0-5,"agreeableness":0-5,"neuroticism":0-5},'
'"social":{"initiative":"主动/被动","conflict":"应对方式","decision":"决策习惯","expression":["句式/口头禅"]},'
'"values":["价值观关键词"],"mbti_guess":"类型","confidence":0-1}'

def label_one(text):
    msgs=[{"role":"system","content":PROMPT},
          {"role":"user","content":f"语料：{text[:1000]}"}]
    inp=tok(tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False),
            return_tensors="pt").to("mps")
    out=m.generate(**inp,max_new_tokens=400,do_sample=False)
    return tok.decode(out[0][inp.input_ids.shape[1]:],skip_special_tokens=True)

# 示例：给前 3 条打标（正式跑改 range(len(segs))）
results=[]
for s in segs[:3]:
    raw=label_one(s["text"])
    results.append({"source":s["file"],"text":s["text"],"raw_label":raw})
    print(s["text"][:30],"->",raw[:120].replace("\n"," "))
with open(os.path.expanduser("~/Desktop/Parallax项目/SML构建部署/语料数据集/自建素材_待审核标签.jsonl"),"w",encoding="utf-8") as f:
    for r in results: f.write(json.dumps(r,ensure_ascii=False)+"\n")

/Users/chris/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/chris/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.14s/it]


In [3]:
# 3) 标签后处理：解析 JSON + 并入统一格式（raw_label -> 结构化字段）
import json, re
def parse(raw):
    i=raw.find("{"); j=raw.rfind("}")
    if i<0 or j<=i: return None
    try: return json.loads(raw[i:j+1])
    except Exception: return None
rows=[]
for r in results:
    obj=parse(r["raw_label"])
    if obj:
        rows.append({"id":f"self_{len(rows):05d}","source":r["source"],"lang":"zh","text":r["text"],
                     "mbti":obj.get("mbti_guess"),"mbti_conf":obj.get("confidence"),
                     "ocean":obj.get("ocean"),"social":obj.get("social"),
                     "label_note":"4B 弱标签，人工抽检 5%","license":"自有授权素材"})
with open(os.path.expanduser("~/Desktop/Parallax项目/SML构建部署/语料数据集/自建素材_已标注.jsonl"),"w",encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r,ensure_ascii=False)+"\n")
print("结构化入库:",len(rows),"条")

结构化入库: 0 条
